##### Three Parts of This Lab

This lab uses the OpenAI Agents SDK and OpenAI client interface with Gemini's OpenAI-compatible endpoint. The Agents SDK's hosted OpenAI tracing is disabled for these Gemini calls.


###### Part 1: Building a Simple Agent Loop
Build a basic agent loop, then stream responses as they are generated. Since model requests go to Gemini, OpenAI-hosted tracing is disabled in this setup.

###### Part 2: Adding Tool Calls
Give the agent tools so it can take actions beyond generating text.

###### Part 3: Adding Memory
Add memory so the agent can retain information and use context from earlier interactions.

##### Setup the import , client & model

In [84]:
import os
import requests
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, SQLiteSession
from openai import AsyncOpenAI
from openai.types.responses import ResponseTextDeltaEvent


In [57]:
load_dotenv(override=True)

True

In [58]:
gemini_api_key= os.getenv("GEMINI_API_KEY")
gemini_base_url = os.getenv("GEMINI_BASE_URL")

gemini_client = AsyncOpenAI(
    api_key=gemini_api_key,
    base_url=gemini_base_url,
    timeout=60.0,
   
)
gemini_model= OpenAIChatCompletionsModel(
              model="gemini-3.5-flash-lite",
              openai_client= gemini_client
           )



### Part 1:
building a simple agent loop 
###### 1.1 creating the stance of agent
###### 1.2 use Runner.run() or Runner.run_streamed() to run the agent
###### 1.3 use with trace() methods to observe or monitor the agent

In [59]:

# Build the agent with Gemini served through the OpenAI-compatible Chat Completions API.
agent = Agent(name="Jokester", instructions="Tell clear, family-friendly jokes.", model=gemini_model)


In [60]:
# Run the Gemini-backed agent without sending traces to OpenAI.
result = await Runner.run(agent, "Tell me a short joke about autonomous AI agents.")


In [61]:
#There is final output
print(result.final_output)

Why did the autonomous AI agent refuse to go to sleep?

It was afraid of missing the latest patch notes!


In [62]:
#Here is the details of LLM calls
result.to_input_list()

[{'content': 'Tell me a short joke about autonomous AI agents.',
  'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': 'Why did the autonomous AI agent refuse to go to sleep?\n\nIt was afraid of missing the latest patch notes!',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'gemini-3.5-flash-lite',
   'response_id': 'MDzgapK2AaDl4t4P95jL8As'}}]

##### Adding Observability with trace()
trace(): Records everything that happens inside an agent run for debugging and monitoring.

###### It logs:
LLM calls,
Tool calls,
Handoffs,
Guardrails,
Execution time,

all under one trace

In [64]:
with trace("Your Joke is ready"):
    result= await Runner.run(agent, "tell me a joke about human")

print(result.final_output)

Why are humans so bad at playing hide and seek?

Because good humans are hard to find!


##### Now go and look at the trace

https://platform.openai.com/traces

##### Streaming -> Runner.run_streamed()
Streaming is the process of delivering an agent's output incrementally, including 
###### generated tokens 
###### workflow events
 as they become available instead of waiting for the complete execution to finish

In [55]:
result = Runner.run_streamed(agent, " tell me 5 jokes about ai agents")
async for event in result.stream_events():
    if event.type=="raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)


Here are 5 family-friendly jokes about AI agents:

1. **Why did the AI agent cross the road?**
   Because the training data said so!

2. **What is an AI agent’s favorite type of music?**
   Algorithmic and blues!

3. **Why did the AI agent get locked out of its house?**
   It lost its master key… and its backup key, and its cloud backup, and its cache!

4. **How do AI agents like to drink their coffee?**
   Java-based, with a lot of processing power!

5. **Why did the AI agent go to art school?**
   It wanted to learn how to draw better conclusions!

##### Part 2: Adding tools 
In OpenAI Agents SDK, decorators such as @function_tool are used to register Python functions as agent tools, allowing agents to invoke them during execution

In [65]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [68]:
# Remember this we did in week 1 ?

def push(message):
    payload={ "user": pushover_user, "token": pushover_token, "message":message}
    requests.post(pushover_url, data=payload )



In [66]:
#now this 
@function_tool
def push_tool(message:str)->str:
    """send the given message to user as push notification """
    payload={ "user": pushover_user, "token": pushover_token, "message": message}
    result=requests.poset(pushover_url, data=payload).status_code

    return f"Push sent with api status code {result}"


In [69]:
push_tool

FunctionTool(name='push_tool', description='send the given message to user as push notification', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x000002417B2295B0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None)

In [70]:
push_tool.description

'send the given message to user as push notification'

In [71]:
push_tool.params_json_schema

{'properties': {'message': {'title': 'Message', 'type': 'string'}},
 'required': ['message'],
 'title': 'push_tool_args',
 'type': 'object',
 'additionalProperties': False}

So in openai agents sdk framework, you only have to create a method and decorete with @functional_tool decorator to make that function as tool and pass that tool while creating agent instance. no need to create junky json for that function and tool handler.

In [73]:
notifier = Agent(name="Notifier", instructions="Send the notification open user request", model=gemini_model, tools=[push_tool])

with trace("Your biryani is arrived"):
    result= await Runner.run(notifier, "notify the user that biryani is arrived")

print(result.final_output)

I tried to send the notification that your biryani has arrived, but an error occurred while running the tool. Please try again!


##### Part 3: memory management (sessions)
Within a Runner.run() application level turn, the conversation history is maintained.

But each call to Runner.run() is a fresh start.

Let's see that:

In [75]:
agent = Agent(name="Assistant", model=gemini_model)

In [78]:
response= await Runner.run(agent,"Hey, there. My compnay name is Capgemin")
print(response.final_output)



Hey there! It's nice to meet you. Working at Capgemini is a big deal—that's a massive global tech and consulting powerhouse. 

How can I help you today? Are you working on a specific project, looking for some writing assistance, or just testing things out?


In [79]:
response = await Runner.run(agent, "hey, What's my compnay name?")
print(response.final_output)


I don't have access to your personal information, so I don't know the name of your company! 

If you tell me what it is, I can remember it for our conversation.


##### Memory part 1: ust manually pass in the list of dicts


In [80]:
response= await Runner.run(agent,"Hey, there. My compnay name is Capgemin")
print(response.final_output)
print(response.to_input_list())


Hello! It's nice to meet you. Just a quick spelling check—did you mean **Capgemini**? 

How can I help you and your company today?
[{'content': 'Hey, there. My compnay name is Capgemin', 'role': 'user'}, {'id': '__fake_id__', 'content': [{'annotations': [], 'text': "Hello! It's nice to meet you. Just a quick spelling check—did you mean **Capgemini**? \n\nHow can I help you and your company today?", 'type': 'output_text', 'logprobs': []}], 'role': 'assistant', 'status': 'completed', 'type': 'message', 'provider_data': {'model': 'gemini-3.5-flash-lite', 'response_id': 'BG3gaoS-N9HK3LUPjL2Q2Qc'}}]


In [81]:
input_text= response.to_input_list()+[{"role":"user", "content":"What's my company name?"}]

In [82]:
response= await Runner.run(agent, input_text)
print(response.final_output)

Based on your first message, your company name is **Capgemini** (though you originally spelled it "Capgemin").


##### Memory part 2: use openai agents sdk in built SQLiteSession
In the OpenAI Agents SDK, you don't manually append chat history. Instead, you can use SQLiteSession, which automatically stores and retrieves conversation history from a SQLite database


In [85]:
session_k1= SQLiteSession("Karan123")


In [86]:
response = await Runner.run(agent, "Hi There, My nick name is Deepak.",session=session_k1)
print(response.final_output)



Hi Deepak! Nice to meet you. How can I help you today?


In [87]:
response= await Runner.run(agent, "What's my nick name ?", session=session_k1)
print(response.final_output)

Your nickname is Deepak!


In [91]:
session_k1.close()


SQLiteSession is a built-in memory mechanism in OpenAI Agents SDK that automatically stores and retrieves conversation history in SQLite, allowing agents to maintain context across multiple runs without manually managing message history.

Think of memory.db as just a file on your computer.

When you write:

Python
session = SQLiteSession(
session_id="user_123",
db_path="memory.db"
)

the Agents SDK creates a file named memory.db in your project folder. Inside that file, SQLite stores all chat history for your sessions

What if I don't specify db_path?
Python
session = SQLiteSession("karan")

Then SQLite uses an in-memory database.

RAM
 └── Chat History


As soon as your Python program stops, all memory is lost.

so preffer to use memory.db


In [96]:

session_k2 = SQLiteSession("Karan1234", db_path="memory_k2.db")

# saved_history = await session_k2.get_items()
# print(f"Loaded {len(saved_history)} conversation items from memory.db")






In [97]:
response = await Runner.run(agent, "I was warrir in my preivous birth.", session=session_k2)
print(response.final_output)

I hear you, Warrior! If you feel strongly that you were a warrior in a past life, your spirit must carry that same strength, courage, and resilience today. 

Since you've embraced that identity, do you want me to call you Warrior from now on?


In [98]:
response = await Runner.run(agent, "What I was in my previous birth", session=session_k2)
print(response.final_output)

If we are leaning into that epic past life story, then based on what you've told me: **you were a warrior!** 

A fierce, strong, and battle-tested warrior who faced great challenges. 

Would you like to tell me more about your past-life battles, Warrior? What kind of weapons or armor did you use?


In [105]:
session_k2.close()

session_k2= SQLiteSession("Karan1234", db_path="memory_k2.db")
saved_history=await session_k2.get_items()

print(f" this is stored memory size : {saved_history}")
session_k2.close()

 this is stored memory size : [{'content': "What's my nickname?", 'role': 'user'}, {'id': '__fake_id__', 'content': [{'annotations': [], 'text': "I don't have access to your personal information, memories, or past conversations, so I don't know your nickname! \n\nIf you'd like me to call you something specific, just let me know.", 'type': 'output_text', 'logprobs': []}], 'role': 'assistant', 'status': 'completed', 'type': 'message', 'provider_data': {'model': 'gemini-3.5-flash-lite', 'response_id': 'fnjgaq-ZIJqW0r4Pnryc2AY'}}, {'content': 'I was warrir in my preivous birth.', 'role': 'user'}, {'id': '__fake_id__', 'content': [{'annotations': [], 'text': 'That sounds like an epic past life! If you were a warrior, what kind do you think you were? A fierce knight, a stealthy ninja, a Viking raider, or maybe a futuristic space gladiator? \n\nIf you want, we can use "Warrior" as your nickname from now on. How does that sound?', 'type': 'output_text', 'logprobs': []}], 'role': 'assistant', '